# Module 01 — AI Workbook Companion

A lightweight companion to [Session1.ipynb](../Session1.ipynb). It does **not**
replace the existing exercises — it adds the supervision layer from
[Module 11](../../11/README.md) to the Module 01 material (kernels, thread
hierarchy, execution configuration, `cudaDeviceSynchronize`, grid-stride loops,
vector add).

Use it after you've done the normal Session 1 exercises. It reframes the same
work as the five-phase loop so that "the AI wrote my kernel" still means "I
learned how the GPU runs it."

## The loop, applied to vector add

Your task: parallelize [01/07-vector-add/01-vector-add.cu](../07-vector-add/01-vector-add.cu)
on the GPU. You may have an AI write the kernel. Your job is everything around it.

1. **SPECIFY** — Before generating anything, write the contract for
   `addVectorsInto` as a kernel: input/output arrays and sizes, where the memory
   lives (`malloc` vs `cudaMallocManaged`), the launch configuration you'll use
   for `N = 2<<20` elements, and how you'll know it's correct
   (`checkElementsAre(7, ...)`).
2. **GENERATE** — Ask the AI to write the kernel and the host launch. Keep its
   output in a separate file; don't hand-fix it silently.
3. **PREDICT** — *Before running:* how many blocks and threads? What's the
   occupancy? Where are the correctness risks? For Module 01, the classic risk is
   **missing `cudaDeviceSynchronize()`** — the CPU races ahead and checks results
   before the kernel finishes.
4. **VERIFY** — Use [verify_vector_add.cu](./verify_vector_add.cu). It gives you a
   CPU reference and a harness *stub*; you complete the launch and the assertions,
   including the synchronization the AI may have dropped.
5. **DIAGNOSE** — Time it, and explain whether it's memory-bound (it is — vector
   add moves 3 floats per add), and what that means for the ceiling.

## The adversarial exercise

[adversarial_vector_add_buggy.cu](./adversarial_vector_add_buggy.cu) is a kernel
"an AI wrote for you." It compiles and *looks* correct. **It is not.** It has the
signature Module 01 bug: the host reads the results without waiting for the GPU.

Your job:

1. Read it and predict what will happen *before* running it.
2. Build and run it (see the commands at the top of the file). Observe the
   failure — it may fail intermittently, which is the whole lesson about races.
3. In one sentence, state the exact bug and the one line that fixes it.
4. Prove your fix: after adding the fix, the check must pass reliably across
   several runs.

Could an AI catch this bug? Sometimes — if you ask it to look specifically for a
missing synchronization, it may spot it. But it cannot be *relied on* to, and it
cannot be held responsible when it misses one: the bug produces no compiler
error and sometimes even returns the right answer, so it can slip past both a
single test run and an AI review. Catching it reliably is your job, and a test
harness you wrote and understand is how you do it.

A worked solution and full explanation are in [solution.ipynb](./solution.ipynb). Try
the exercise yourself first.

## Reflection prompt (write this down)

- The AI's kernel compiled and sometimes gave the right answer. What made it
  wrong anyway, and why would a test that runs once have missed it?
- Vector add barely computes anything per element. Is the GPU version faster
  because of compute or because of memory bandwidth? What does that predict about
  problems where the arithmetic per element is also tiny?
- If you had shipped the AI's kernel without `cudaDeviceSynchronize()`, when
  would the bug have surfaced — in your test, or in production?


In [ ]:
# Prerequisites: verify the GPU toolchain before running this workbook.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

## Step 1 - Reproduce the bug

Build and run the problem program [adversarial_vector_add_buggy.cu](./adversarial_vector_add_buggy.cu). Read it first and predict what will happen before you run this cell.

In [ ]:
!nvcc -arch=native adversarial_vector_add_buggy.cu -o buggy && ./buggy

## Step 2 - Run your verification harness

[verify_vector_add.cu](./verify_vector_add.cu) ships a CPU reference and a PASS/FAIL gate, but it **defaults to FAIL** until you complete the `TODO` sections in it (launch, error check, synchronization). Edit the file, then re-run this cell until it prints `PASS`.

In [ ]:
!nvcc -arch=native verify_vector_add.cu -o verify && ./verify

## Next

Once your harness prints `PASS`, open **[solution.ipynb](./solution.ipynb)** to compare your fix with the worked answer and explanation. See [Module 11](../../11/README.md) for the method behind this workbook.